# Data Cleaning Notebook

Within this notebook, a core snRNA-seq retina dataset is split between neuronal and non-neuronal cell types and downsampled. The resulting datasets are stored in the same directory as the original. 

* Data must be downsampled if it contains more than 500,000 cells. If downsampled, the resulting datasets are under `data_{structure}_ds.h5ad`. 
* Data must be filtered if it contains unknown or diseased cells. If filtered, the resulting datasets are under `data_{structure}_filter.h5ad`.
* If stratified, the resulting datasets are under `data_{structure}_{strata}.h5ad`.

In this manner, the tag hierarchy goes `data_{structure}_ds_filter_{strata}.h5ad`. 

**Note:** The input data file should be a single-cell RNA-seq dataset in h5ad format, with cell type annotations in the obs dataframe and gene annotations in the var dataframe.

**Note:** If using GitHub for version control and repository sharing, ensure that you add the path to your data folder to the repository's `.gitignore` file, to prevent yourself from exceeding the GitHub's storage limits.

## Configs

In [1]:
# libraries
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import celltypist as ct

/opt/miniconda3/envs/nsforest/lib/python3.11/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [ ]:
# configs
code_folder = "/Users/vbecker/NSForest-ncRNA" # path to the NSForest-ncRNA folder
sys.path.insert(0, os.path.abspath(code_folder))

data_folder = "../beckersv_data/" # path to folder containing the input data file (.h5ad format)
output_folder = "../beckersv_data_clean/" # path to folder where output files will be written

to_downsample_ideal = 500000 # ideal number of cells within each strata's final .h5ad file.

seed = 0 # random seed for reproducibility

## Functions

In [3]:
def find_threshold(og_anndata, cluster_header, ideal_total):
    """
    Args:
        og_anndata (adata): Anndata object containing the original dataset
        cluster_header (str): Header of the column in og_anndata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get counts dataframe
    og_counts = pd.DataFrame(og_anndata.obs[cluster_header].value_counts()).reset_index()
    print("Original Counts:\n" + str(og_counts))
    
    # if there's less cells than the ideal total, just return the number of cells
    if og_counts['count'].sum() <= ideal_total:
        return ideal_total

    # set upper and lower limits for binary search
    upper_lim = ideal_total
    lower_lim = int(ideal_total / len(og_counts.index))
    
    # recurse
    return find_threshold_recursive(og_counts, ideal_total, upper_lim, lower_lim)
    
def find_threshold_recursive(count_data, ideal_total, upper_lim, lower_lim):
    """
    Args:
        count_data (pd.DataFrame): Dataframe containing the counts of each cluster
        ideal_total (int): Ideal total number of cells to downsample to
        upper_lim (int): Upper limit for the binary search
        lower_lim (_type_): Lower limit for the binary search

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get middle lim
    mid_thresh = (upper_lim + lower_lim) // 2
    
    # get new total
    new_counts = count_data['count'].clip(upper=mid_thresh)
    new_total = new_counts.sum()
    
    # base case check
    if abs(new_total - ideal_total) <= ideal_total * 0.02:
        print("Ideal threshold found!")
        print(f"Ideal Threshold : {mid_thresh}, New Total : {new_total}, Ideal Total : {ideal_total}.")
        print("New Counts:\n" + str(new_counts))
        return mid_thresh
    
    # binary search exhausted :[
    if upper_lim - lower_lim <= 1:
        print("Binary search exhausted :[")
        print(f"Ideal Threshold : {mid_thresh}, New Total : {new_total}, Ideal Total : {ideal_total}.")
        print("New Counts:\n" + str(new_counts))
        return mid_thresh

    # sending the recursive case
    if new_total > ideal_total:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            mid_thresh,
            lower_lim
        )
    else:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            upper_lim,
            mid_thresh
        )
    

In [4]:
def downsample(adata, cluster_header, ideal_total, seed, filepath=None, filename=None, return_index=False, write_to_file=True):
    """
    Args:
        adata (ad.AnnData): Anndata object containing the dataset to downsample
        cluster_header (str): Header of the column in adata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to
        seed (int): Random seed for reproducibility
        filepath (str | None): Path to the folder where the downsampled anndata object will be saved
        filename (str | None): Name of the file where the downsampled anndata object will be saved
        return_index (bool, optional): Whether to return the indices of the downsampled cells. Defaults to False.
        write_to_file (bool, optional): Whether to write the downsampled anndata object to a .h5ad file. Defaults to True.
        
    Returns:
        np.ndarray: Indices of the downsampled cells if return_index is True, otherwise None
    """
    
    if not (return_index or write_to_file):
        raise ValueError("At least one of return_index or write_to_file must be True.") 
    
    if write_to_file and (filepath is None or filename is None):
        raise ValueError("If write_to_file is True, both filepath and filename must be provided.")
    
    # find threshold
    print(f"Finding threshold for downsampling to {ideal_total} cells...")
    thresh = find_threshold(adata, cluster_header, ideal_total)
    
    # downsample for all cell types
    print(f"Downsampling to maximum {thresh} cells per cluster...")
    idx = ct.samples.downsample_adata(
        adata,
        mode="each",
        by=cluster_header,
        n_cells=thresh,
        random_state=seed,
        return_index=True,
    )
    
    # write to file if requested
    if write_to_file:
        print(f"Writing downsampled anndata object to {filepath + filename}...")
        adata[idx, :].to_memory().write_h5ad(filename = filepath + filename)
        
    # return index if requested
    if return_index:
        print(f"Returning indices of downsampled cells...")
        return idx

In [ ]:
# annotate adata with gene types (protein coding, lncRNA, etc.) by ENSEMBL id
import bionty_base as bt
gene_bt = bt.Gene(source = "ensembl", version = "release-110")
gene_df = gene_bt.df()
gene_df = gene_df[["ensembl_gene_id", "symbol", "biotype"]]

def annotate_gene_types(adata):
    """
    Args:
        adata (AnnData): AnnData object containing the dataset to annotate

    Returns:
        AnnData: AnnData object with gene type annotations added to adata.var
    """
    
    # DIAGNOSIS!
    ids = set(adata.var["gene"])
    bt_ids = set(gene_df["symbol"])

    print("Number of AnnData Genes:", len(ids))
    print("Union of AnnData Genes and BT Genes:", len(ids & bt_ids))
    print("Number of AnnData Genes Not In BT Genes: ", len(ids - bt_ids))
    print(len(ids - bt_ids))
    
    # merge from gene_df
    gene_df.drop_duplicates(subset=["symbol"], inplace=True)
    adata.var = adata.var.merge(gene_df, left_on="gene", right_on="symbol", how="left")
    
    # merge from gencode annotations
    gencode47_df = pd.read_csv(data_folder + "gencode_v47_annotation.csv", sep=",", header=0) # ENSEMBL_ID, gene_type
    gencode47_df.drop_duplicates(subset=["gene_name"], inplace=True)
    gencodevM36_df = pd.read_csv(data_folder + "gencode_vM36_annotation.csv", sep=",", header=0) # ENSEMBL_ID, gene_type
    gencodevM36_df.drop_duplicates(subset=["gene_name"], inplace=True)
    
    # rename columns to match adata.var
    gencode47_df.rename(columns={"gene_type": "biotype"}, inplace=True)
    gencodevM36_df.rename(columns={"gene_type": "biotype"}, inplace=True)
    
    # merge gencode annotations
    adata.var = adata.var.merge(gencode47_df, left_on="gene", right_on="gene_name", how="left", suffixes=("", "_gencode47"))
    adata.var = adata.var.merge(gencodevM36_df, left_on="gene", right_on="gene_name", how="left", suffixes=("", "_gencodevM36"))
    
    # merge columns to create a final biotype column
    adata.var["biotype"] = adata.var["biotype"].combine_first(adata.var["biotype_gencode47"]).combine_first(adata.var["biotype_gencodevM36"])
    
    # check for missing gene types
    missing_gene_types = adata.var["biotype"].isnull().sum()
    if missing_gene_types > 0:
        print(f"Warning: {missing_gene_types} genes are missing gene type annotations.")
        missing = adata.var.loc[
            adata.var["biotype"].isna(),
            "gene"
        ]
        missing.to_csv(output_folder + "missing_gene_types.csv", index=False)
    
    return adata

## Testing Gene Annotation (Unused)

In [56]:
import pandas as pd
import mygene

mg = mygene.MyGeneInfo()

df = pd.read_csv(data_folder + "missing_gene_types.csv")   # column: gene_symbol

results = mg.querymany(
    df["gene"].tolist(),
    scopes="symbol",
    fields=[
        "symbol",
        "name",
        "type_of_gene",
        "entrezgene",
        "taxid"
    ],
    species="human",
    as_dataframe=False,
)

result_df = pd.DataFrame(results)

# merge back
annotated = df.merge(
    result_df[["query", "symbol", "name", "type_of_gene", "notfound"]],
    left_on="gene",
    right_on="query",
    how="left"
)

annotated["status"] = "found"
annotated.loc[annotated["notfound"] == True, "status"] = "not found"

querying 1-1000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 1001-2000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 2001-3000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 3001-4000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 4001-5000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 5001-6000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 6001-7000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 7001-8000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 8001-9000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 9001-10000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 10001-11000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 1

In [57]:
print(annotated['type_of_gene'].value_counts())
print(annotated['status'].value_counts())

type_of_gene
ncRNA             6151
pseudo            2537
protein-coding     103
snRNA               28
snoRNA              15
other               13
tRNA                 3
unknown              2
Name: count, dtype: int64
status
not found    10678
found         8859
Name: count, dtype: int64


## Cleaning Template

In [ ]:
# load the dataset
file = "*.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

In [ ]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

In [ ]:
# cell counts by cell type
print(adata_raw.obs["*"].value_counts()) 

In [ ]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

In [ ]:
# gene counts by feature type
print(adata_raw.var["*"].value_counts())

In [ ]:
# defining cluster header
cluster_header = "*" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [ ]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

In [ ]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells

unknown_col = "unknown"          # column name in adata_raw.obs that contains the unknown labels
unknown_values = ["unknown"]     # values in the unknown_col that indicate unknown cells
unhealthy_col = "unhealthy"      # column name in adata_raw.obs that contains the unhealthy labels
unhealthy_values = ["unhealthy"] # values in the unhealthy_col that indicate unhealthy cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unknown_col].isin(unknown_values)
    | adata_raw.obs[unhealthy_col].isin(unhealthy_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_str_ds_filter.h5ad")

In [ ]:
# if not applicable, just downsample by cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, output_folder, "data_str_ds.h5ad", return_index=False, write_to_file=True)

## Motor Cortex (TBD)

* **Original Dataset Size:** 181.6k cells

* **Number of Cell Types:** 127

In [118]:
# load the dataset
file = "data_m1.h5ad"
adata_raw = sc.read_h5ad(data_folder + file)
adata_raw

AnnData object with n_obs × n_vars = 76533 × 50281
    obs: 'cluster_id', 'class', 'subclass', 'self_reported_sex', 'anatomical_region', 'cortical_layer', 'cell_type_accession', 'cell_type_alias', 'cell_type_alt_alias', 'cell_type_designation', 'donor_id', 'load_id', 'assay', 'suspension_type', 'is_primary_data', 'self_reported_ethnicity', 'disease', 'organism', 'organism_ontology_term_id', 'assay_ontology_term_id', 'anatomical_region_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'self_reported_sex_ontology_term_id', 'brain_region_ontology_term_id', 'brain_region'
    var: 'gene', 'highly_variable_genes_standard', 'ensembl_id'
    uns: 'cellSet', 'clusterStatsColumns', 'cluster_info', 'default_embedding', 'dend', 'filter', 'hierarchy', 'mapmycells', 'mode', 'schema_version', 'taxonomyDir', 'title'
    obsm: 'X_default_standard'
    varm: 'cluster_id_median_expr_standard'

In [120]:
# Convert to memory before checking
print(adata_raw.X)

None


In [80]:
# annotate gene types
adata_raw = annotate_gene_types(adata_raw)

50281
30696
19585


/var/folders/jm/kqc7wh8d1n14q977rv6gcdbh0000gn/T/ipykernel_88908/4001739558.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_df.drop_duplicates(subset=["symbol"], inplace=True)
/var/folders/jm/kqc7wh8d1n14q977rv6gcdbh0000gn/T/ipykernel_88908/4001739558.py:29: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  gencode47_df = pd.read_csv(data_folder + "gencode_v47_annotation.csv", sep=",", header=0) # ENSEMBL_ID, gene_type


In [81]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['cluster_id', 'class', 'subclass', 'self_reported_sex',
       'anatomical_region', 'cortical_layer', 'cell_type_accession',
       'cell_type_alias', 'cell_type_alt_alias', 'cell_type_designation',
       'donor_id', 'load_id', 'assay', 'suspension_type', 'is_primary_data',
       'self_reported_ethnicity', 'disease', 'organism',
       'organism_ontology_term_id', 'assay_ontology_term_id',
       'anatomical_region_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id',
       'self_reported_sex_ontology_term_id', 'brain_region_ontology_term_id',
       'brain_region'],
      dtype='object')

In [82]:
# cell counts by cell type
print(adata_raw.obs["cluster_id"].value_counts()) 

cluster_id
Exc L2 LINC00507 GLRA3       12485
Exc L3-5 RORB LNX2            4631
Exc L3 LAMP5 CARM1P1          4044
Exc L2-3 RORB CCDC68          3331
Inh L2-5 PVALB RPH3AL         3253
                             ...  
VLMC L1-5 PDGFRA COLEC12        40
Inh L1 SST P4HA3                38
Inh L5-6 SST DNAJC14            35
Exc L6 FEZF2 PDYN               22
Oligo L5-6 OPALIN LDLRAP1        6
Name: count, Length: 127, dtype: int64


In [83]:
# gene exploration 
print(adata_raw.var['gene']) # gene names
print(adata_raw.var.columns) # gene metadata

0          DDX11L1
1           WASH7P
2        MIR6859-1
3        MIR1302-2
4          FAM138A
           ...    
50276          ND6
50277         TRNE
50278         CYTB
50279         TRNT
50280         TRNP
Name: gene, Length: 50281, dtype: object
Index(['gene', 'highly_variable_genes_standard', 'ensembl_id',
       'ensembl_gene_id', 'symbol', 'biotype', 'Chromosome', 'Source',
       'Feature', 'Start', 'End', 'Score', 'Strand', 'Frame', 'gene_id',
       'biotype_gencode47', 'gene_name', 'level', 'tag', 'transcript_id',
       'transcript_type', 'transcript_name', 'exon_number', 'exon_id',
       'transcript_support_level', 'havana_transcript', 'hgnc_id',
       'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl', 'length',
       'ENSEMBL_ID', 'tx_length', 'Chromosome_gencodevM36',
       'Source_gencodevM36', 'Feature_gencodevM36', 'Start_gencodevM36',
       'End_gencodevM36', 'Score_gencodevM36', 'Strand_gencodevM36',
       'Frame_gencodevM36', 'gene_id_gencodevM36', '

In [88]:
# gene counts by feature type
print(adata_raw.var["biotype"].value_counts())
print(adata_raw.var["biotype"].value_counts().sum())

biotype
protein_coding                        18105
processed_pseudogene                   4994
lncRNA                                 2032
miRNA                                  1849
unprocessed_pseudogene                 1162
transcribed_unprocessed_pseudogene      524
rRNA_pseudogene                         431
snoRNA                                  372
transcribed_processed_pseudogene        255
snRNA                                   199
IG_V_pseudogene                         159
IG_V_gene                               129
TR_V_gene                               106
TR_J_gene                                79
transcribed_unitary_pseudogene           70
misc_RNA                                 60
unitary_pseudogene                       56
TR_V_pseudogene                          33
IG_D_gene                                27
rRNA                                     20
IG_J_gene                                18
scaRNA                                   18
IG_C_gene               

In [85]:
# defining cluster header
cluster_header = "cluster_id" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [86]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,cluster_id,count
0,Exc L2 LINC00507 GLRA3,12485
1,Exc L3-5 RORB LNX2,4631
2,Exc L3 LAMP5 CARM1P1,4044
3,Exc L2-3 RORB CCDC68,3331
4,Inh L2-5 PVALB RPH3AL,3253
...,...,...
122,VLMC L1-5 PDGFRA COLEC12,40
123,Inh L1 SST P4HA3,38
124,Inh L5-6 SST DNAJC14,35
125,Exc L6 FEZF2 PDYN,22


In [87]:
adata_raw._raw = None  # fix an issue with original data that prevented writing

In [ ]:
# if not applicable, just downsample by cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, output_folder, "data_m1_ds.h5ad", return_index=False, write_to_file=True)

Finding threshold for downsampling to 500000 cells...
Original Counts:
                    cluster_id  count
0       Exc L2 LINC00507 GLRA3  12485
1           Exc L3-5 RORB LNX2   4631
2         Exc L3 LAMP5 CARM1P1   4044
3         Exc L2-3 RORB CCDC68   3331
4        Inh L2-5 PVALB RPH3AL   3253
..                         ...    ...
122   VLMC L1-5 PDGFRA COLEC12     40
123           Inh L1 SST P4HA3     38
124       Inh L5-6 SST DNAJC14     35
125          Exc L6 FEZF2 PDYN     22
126  Oligo L5-6 OPALIN LDLRAP1      6

[127 rows x 2 columns]
Downsampling to maximum 500000 cells per cluster...
Writing downsampled anndata object to ../beckersv_data/data_m1_ds.h5ad...


KeyError: "Unable to synchronously open object (object 'X' doesn't exist)"

## Middle Temporal Gyrus (TBD)

* **Original Dataset Size:** 15.9k cells

* **Number of Cell Types:** 75

In [124]:
# load the dataset
file = "data_mtg.h5ad"
adata_raw = sc.read_h5ad(data_folder + file)
adata_raw

AnnData object with n_obs × n_vars = 15603 × 50281
    obs: 'sample_id', 'sample_type', 'organism', 'donor_id', 'self_reported_sex', 'age_days', 'brain_hemisphere', 'brain_region', 'brain_subregion', 'facs_date', 'facs_container', 'facs_sort_criteria', 'rna_amplification_set', 'library_prep_set', 'library_prep_avg_size_bp', 'seq_name', 'seq_tube', 'seq_batch', 'total_reads', 'percent_exon_reads', 'percent_intron_reads', 'percent_intergenic_reads', 'percent_rrna_reads', 'percent_mt_exon_reads', 'percent_reads_unique', 'percent_synth_reads', 'percent_ecoli_reads', 'percent_aligned_reads_total', 'complexity_cg', 'genes_detected_cpm_criterion', 'genes_detected_fpkm_criterion', 'class', 'cluster_id', 'subclass', 'load_id', 'assay', 'anatomical_region', 'suspension_type', 'is_primary_data', 'self_reported_ethnicity', 'disease', 'organism_ontology_term_id', 'assay_ontology_term_id', 'anatomical_region_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', '

In [130]:
# Convert to memory before checking
adata_raw.layers.keys()

KeysView(Layers with keys: )

In [ ]:
# annotate gene types
adata_raw = annotate_gene_types(adata_raw)

50281
30696
19585


/var/folders/jm/kqc7wh8d1n14q977rv6gcdbh0000gn/T/ipykernel_88908/4001739558.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_df.drop_duplicates(subset=["symbol"], inplace=True)
/var/folders/jm/kqc7wh8d1n14q977rv6gcdbh0000gn/T/ipykernel_88908/4001739558.py:29: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  gencode47_df = pd.read_csv(data_folder + "gencode_v47_annotation.csv", sep=",", header=0) # ENSEMBL_ID, gene_type


In [ ]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['cluster_id', 'class', 'subclass', 'self_reported_sex',
       'anatomical_region', 'cortical_layer', 'cell_type_accession',
       'cell_type_alias', 'cell_type_alt_alias', 'cell_type_designation',
       'donor_id', 'load_id', 'assay', 'suspension_type', 'is_primary_data',
       'self_reported_ethnicity', 'disease', 'organism',
       'organism_ontology_term_id', 'assay_ontology_term_id',
       'anatomical_region_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id',
       'self_reported_sex_ontology_term_id', 'brain_region_ontology_term_id',
       'brain_region'],
      dtype='object')

In [ ]:
# cell counts by cell type
print(adata_raw.obs["cluster_id"].value_counts()) 

cluster_id
Exc L2 LINC00507 GLRA3       12485
Exc L3-5 RORB LNX2            4631
Exc L3 LAMP5 CARM1P1          4044
Exc L2-3 RORB CCDC68          3331
Inh L2-5 PVALB RPH3AL         3253
                             ...  
VLMC L1-5 PDGFRA COLEC12        40
Inh L1 SST P4HA3                38
Inh L5-6 SST DNAJC14            35
Exc L6 FEZF2 PDYN               22
Oligo L5-6 OPALIN LDLRAP1        6
Name: count, Length: 127, dtype: int64


In [ ]:
# gene exploration 
print(adata_raw.var['gene']) # gene names
print(adata_raw.var.columns) # gene metadata

0          DDX11L1
1           WASH7P
2        MIR6859-1
3        MIR1302-2
4          FAM138A
           ...    
50276          ND6
50277         TRNE
50278         CYTB
50279         TRNT
50280         TRNP
Name: gene, Length: 50281, dtype: object
Index(['gene', 'highly_variable_genes_standard', 'ensembl_id',
       'ensembl_gene_id', 'symbol', 'biotype', 'Chromosome', 'Source',
       'Feature', 'Start', 'End', 'Score', 'Strand', 'Frame', 'gene_id',
       'biotype_gencode47', 'gene_name', 'level', 'tag', 'transcript_id',
       'transcript_type', 'transcript_name', 'exon_number', 'exon_id',
       'transcript_support_level', 'havana_transcript', 'hgnc_id',
       'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl', 'length',
       'ENSEMBL_ID', 'tx_length', 'Chromosome_gencodevM36',
       'Source_gencodevM36', 'Feature_gencodevM36', 'Start_gencodevM36',
       'End_gencodevM36', 'Score_gencodevM36', 'Strand_gencodevM36',
       'Frame_gencodevM36', 'gene_id_gencodevM36', '

In [ ]:
# gene counts by feature type
print(adata_raw.var["biotype"].value_counts())
print(adata_raw.var["biotype"].value_counts().sum())

biotype
protein_coding                        18105
processed_pseudogene                   4994
lncRNA                                 2032
miRNA                                  1849
unprocessed_pseudogene                 1162
transcribed_unprocessed_pseudogene      524
rRNA_pseudogene                         431
snoRNA                                  372
transcribed_processed_pseudogene        255
snRNA                                   199
IG_V_pseudogene                         159
IG_V_gene                               129
TR_V_gene                               106
TR_J_gene                                79
transcribed_unitary_pseudogene           70
misc_RNA                                 60
unitary_pseudogene                       56
TR_V_pseudogene                          33
IG_D_gene                                27
rRNA                                     20
IG_J_gene                                18
scaRNA                                   18
IG_C_gene               

In [ ]:
# defining cluster header
cluster_header = "cluster_id" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [ ]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,cluster_id,count
0,Exc L2 LINC00507 GLRA3,12485
1,Exc L3-5 RORB LNX2,4631
2,Exc L3 LAMP5 CARM1P1,4044
3,Exc L2-3 RORB CCDC68,3331
4,Inh L2-5 PVALB RPH3AL,3253
...,...,...
122,VLMC L1-5 PDGFRA COLEC12,40
123,Inh L1 SST P4HA3,38
124,Inh L5-6 SST DNAJC14,35
125,Exc L6 FEZF2 PDYN,22


In [ ]:
adata_raw._raw = None  # fix an issue with original data that prevented writing

In [ ]:
# if not applicable, just downsample by cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, output_folder, "data_mtg_ds.h5ad", return_index=False, write_to_file=True)

Finding threshold for downsampling to 500000 cells...
Original Counts:
                    cluster_id  count
0       Exc L2 LINC00507 GLRA3  12485
1           Exc L3-5 RORB LNX2   4631
2         Exc L3 LAMP5 CARM1P1   4044
3         Exc L2-3 RORB CCDC68   3331
4        Inh L2-5 PVALB RPH3AL   3253
..                         ...    ...
122   VLMC L1-5 PDGFRA COLEC12     40
123           Inh L1 SST P4HA3     38
124       Inh L5-6 SST DNAJC14     35
125          Exc L6 FEZF2 PDYN     22
126  Oligo L5-6 OPALIN LDLRAP1      6

[127 rows x 2 columns]
Downsampling to maximum 500000 cells per cluster...
Writing downsampled anndata object to ../beckersv_data/data_m1_ds.h5ad...


KeyError: "Unable to synchronously open object (object 'X' doesn't exist)"

## Retina (Downsampled/Stratified)

* **Original Dataset Size:** 3.2M cells

* **Number of Cell Types:** 123

* Stratifying by `majorclass` into neuronal and non-neuronal cell types.

In [25]:
# retina-specific configs
majorclass_values = {
    "neuron": ["AC", # amacrine cell
               "BC", # bipolar cell
               "Cone", 
               "HC", # horizontal cell
               "RGC", # retinal ganglion cell
               "Rod"],
    
    "non-neuron": ["Astrocyte",
                   "MG", # Müller glia
                   "Microglia",
                   "RPE"] # retinal pigment epithelium
}

In [26]:
# load the retina dataset
file = "data_retina_sn.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 3177310 × 35475 backed at '../beckersv_data/data_retina_sn.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', '

In [27]:
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['reference_genome', 'gene_annotation_version', 'alignment_software',
       'intronic_reads_counted', 'donor_id', 'donor_age',
       'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death',
       'donor_living_at_sample_collection', 'sample_id',
       'sample_preservation_method', 'tissue_ontology_term_id',
       'development_stage_ontology_term_id', 'sample_collection_method',
       'tissue_source', 'tissue_type', 'suspension_derivation_process',
       'suspension_dissociation_reagent', 'suspension_enriched_cell_types',
       'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type',
       'tissue_handling_interval', 'library_id', 'assay_ontology_term_id',
       'sequenced_fragment', 'institute', 'library_id_repository',
       'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id',
       'author_cell_type', 'disease_ontology_term_id', 'reported_diseases',
       'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster',


In [28]:
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241860',
       'ENSG00000241599', 'ENSG00000286448', 'ENSG00000236601',
       'ENSG00000284733',
       ...
       'ENSG00000275249', 'ENSG00000274792', 'ENSG00000274175',
       'ENSG00000275869', 'ENSG00000273554', 'ENSG00000277836',
       'ENSG00000278633', 'ENSG00000276017', 'ENSG00000278817',
       'ENSG00000277196'],
      dtype='object', length=35475)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [29]:
print(adata_raw.obs["author_cell_type"].value_counts()) # gene counts by cell type

author_cell_type
Rod       1066056
MG         221612
MG_OFF     200402
MG_ON      151685
FMB        144086
           ...   
HAC90          79
HAC91          74
HAC92          65
HAC93          55
HAC95          39
Name: count, Length: 123, dtype: int64


In [30]:
print(adata_raw.var["feature_type"].value_counts()) # gene counts by feature type

feature_type
protein_coding                        19266
lncRNA                                15491
IG_V_pseudogene                         187
IG_V_gene                               146
TR_V_gene                               106
TR_J_gene                                79
IG_D_gene                                37
TR_V_pseudogene                          33
transcribed_unprocessed_pseudogene       29
transcribed_unitary_pseudogene           19
IG_J_gene                                18
artifact                                 17
IG_C_gene                                14
IG_C_pseudogene                           9
TR_C_gene                                 6
TR_J_pseudogene                           4
TR_D_gene                                 4
processed_pseudogene                      3
IG_J_pseudogene                           3
transcribed_processed_pseudogene          2
TEC                                       1
unprocessed_pseudogene                    1
Name: count, dtype:

In [31]:
cluster_header = "author_cell_type" # column name in adata.obs that contains the cluster labels
                                    # used in NS-Forest
                                    
strata_header = "majorclass" # column name in adata.obs that contains the strata labels

In [32]:
# check cell counts by cluster and strata
print(pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index())
print(pd.DataFrame(adata_raw.obs[strata_header].value_counts()).reset_index())

    author_cell_type    count
0                Rod  1066056
1                 MG   221612
2             MG_OFF   200402
3              MG_ON   151685
4                FMB   144086
..               ...      ...
118            HAC90       79
119            HAC91       74
120            HAC92       65
121            HAC93       55
122            HAC95       39

[123 rows x 2 columns]
  majorclass    count
0        Rod  1066056
1         BC   691008
2         AC   571579
3        RGC   399605
4         MG   221612
5       Cone   127060
6         HC    80548
7  Astrocyte    14085
8  Microglia     4894
9        RPE      863


In [33]:
# compute neuron mask on the adata_raw object
neuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["neuron"])
neuron_idx = np.flatnonzero(neuron_mask)

# temporary backed view
adata_neuron = adata_raw[neuron_mask, :]

In [ ]:
# compute non-neuron mask on the adata_raw object
nonneuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["non-neuron"])
nonneuron_idx = np.flatnonzero(nonneuron_mask)

# temporary backed view
adata_nonneuron = adata_raw[nonneuron_mask, :]

In [34]:
adata_raw._raw = None # fix an issue with original data that prevented writing

In [ ]:
# downsample for all cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, output_folder, "data_retina_all_ds.h5ad", return_index=False, write_to_file=True)

In [ ]:
# downsample for neuron cell types
idx = downsample(adata_neuron, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_retina_ds_neuron.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
    author_cell_type    count
0                Rod  1066056
1             MG_OFF   200402
2              MG_ON   151685
3                FMB   144086
4            ML_Cone   118559
..               ...      ...
114            HAC90       79
115            HAC91       74
116            HAC92       65
117            HAC93       55
118            HAC95       39

[119 rows x 2 columns]
Ideal threshold found!
Ideal Threshold : 7589, New Total : 494316, Ideal Total : 500000.
New Counts:
0      7589
1      7589
2      7589
3      7589
4      7589
       ... 
114      79
115      74
116      65
117      55
118      39
Name: count, Length: 119, dtype: int64
Downsampling to maximum 7589 cells per cluster...
Returning indices of downsampled cells...


In [ ]:
# downsample for non-neuron cell types
idx = downsample(adata_nonneuron, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = nonneuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_retina_ds_nonneuron.h5ad")

## Spinal Cord (Filter)

* **Original Dataset Size:** 67.7k cells

* **Number of Cell Types:** 43

* Filtering out `disease` where values are `amyotrophic lateral sclerosis`.

In [22]:
# load the dataset
file = "data_spc.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 62711 × 35467 backed at '../beckersv_data/data_spc.h5ad'
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'PrimaryAnnotation', 'ThirdAnnotation', 'percent.mt', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_umap_HM_01'

In [23]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['donor_id', 'development_stage_ontology_term_id',
       'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id',
       'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id',
       'cell_type_ontology_term_id', 'assay_ontology_term_id',
       'suspension_type', 'PrimaryAnnotation', 'ThirdAnnotation', 'percent.mt',
       'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue',
       'self_reported_ethnicity', 'development_stage', 'observation_joinid'],
      dtype='object')

In [10]:
# cell counts by cell type
adata_raw.obs["ThirdAnnotation"].value_counts()

ThirdAnnotation
Oligo_3          14659
Oligo_2           9493
Oligo_1           6576
Astro_1           6286
Opc_1             4206
Oligo_4           3721
Astro_2           3497
Micro_1           2248
Astro_3           1602
Micro_2           1208
Micro_3           1091
Micro_4            720
Micro_5            632
EX_9               543
EX_13              411
INH_6              393
Endothelial_1      386
Macro_1            379
Micro_6            314
Meninges_2         314
Lymphocyte_1       289
INH_4              282
INH_1              274
EX_4               253
EX_8               248
EX_12              243
INH_5              218
Pericyte_1         200
OligoProg_1        190
INH_3              178
EX_5               174
MN_1               170
EX_2               169
INH_7              146
EX_10              146
EX_11              136
EX_3               134
Ependymal_1        120
Meninges_1         116
EX_6               106
EX_1                83
EX_7                82
INH_2             

In [11]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241860',
       'ENSG00000241599', 'ENSG00000286448', 'ENSG00000236601',
       'ENSG00000284733',
       ...
       'ENSG00000275249', 'ENSG00000274792', 'ENSG00000274175',
       'ENSG00000275869', 'ENSG00000273554', 'ENSG00000277836',
       'ENSG00000278633', 'ENSG00000276017', 'ENSG00000278817',
       'ENSG00000277196'],
      dtype='object', name='ensembl_id', length=35467)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [13]:
# gene counts by feature type
print(adata_raw.var["feature_type"].value_counts())

feature_type
protein_coding                        19261
lncRNA                                15488
IG_V_pseudogene                         187
IG_V_gene                               146
TR_V_gene                               106
TR_J_gene                                79
IG_D_gene                                37
TR_V_pseudogene                          33
transcribed_unprocessed_pseudogene       29
transcribed_unitary_pseudogene           19
IG_J_gene                                18
artifact                                 17
IG_C_gene                                14
IG_C_pseudogene                           9
TR_C_gene                                 6
TR_J_pseudogene                           4
TR_D_gene                                 4
processed_pseudogene                      3
IG_J_pseudogene                           3
transcribed_processed_pseudogene          2
TEC                                       1
unprocessed_pseudogene                    1
Name: count, dtype:

In [14]:
# defining cluster header
cluster_header = "ThirdAnnotation" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [15]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,ThirdAnnotation,count
0,Oligo_3,14659
1,Oligo_2,9493
2,Oligo_1,6576
3,Astro_1,6286
4,Opc_1,4206
5,Oligo_4,3721
6,Astro_2,3497
7,Micro_1,2248
8,Astro_3,1602
9,Micro_2,1208


In [16]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells
unhealthy_col = "disease"      # column name in adata_raw.obs that contains the unhealthy labels
unhealthy_values = ["amyotrophic lateral sclerosis"] # values in the unhealthy_col that indicate unhealthy cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unhealthy_col].isin(unhealthy_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_spc_ds.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
   ThirdAnnotation  count
0          Oligo_3   7002
1          Oligo_2   4013
2          Oligo_1   3214
3          Oligo_4   2500
4          Astro_1   2459
5            Opc_1   2000
6          Astro_2   1442
7          Micro_1    964
8          Astro_3    529
9          Micro_2    472
10         Micro_3    432
11         Micro_4    291
12         Micro_5    216
13            EX_9    203
14   Endothelial_1    189
15      Meninges_2    165
16           EX_13    163
17           INH_6    162
18         Macro_1    160
19    Lymphocyte_1    155
20            EX_4    141
21            EX_8    123
22           INH_4    121
23           INH_1    119
24         Micro_6    116
25           EX_12    112
26            EX_5    103
27      Pericyte_1    101
28      Meninges_1     93
29     OligoProg_1     86
30           INH_5     83
31            EX_2     78
32           EX_11     77
33            EX_3     76
34            MN_1 

## Breast (Downsample/Filter)

* **Original Dataset Size:** 803.2k cells

* **Number of Cell Types:** 44

* Filtering out `level2` where values are `stripped_nuclei` and `Doublet`.

In [6]:
# load the dataset
file = "data_breast.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 803283 × 33604 backed at '../beckersv_data/data_breast.h5ad'
    obs: 'sampleID', 'sample_type_coarse', 'sample_type', 'processing_date', 'dissociation_minutes', 'parity', 'brca_status', 'condition', 'tissue_condition', 'reported_ethnicity', 'BMI', 'prob_spikein', 'prob_spikein_dblt', 'pred_spikein', 'n_genes', 'percent_mito', 'n_counts', 'level0_global', 'level1_global', 'level0', 'level1', 'level2', 'scrublet_score', 'scrublet_cluster_score', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'donor_age', 'risk_status', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_types', 'genome', 'n_cells', 'highly_variable', 'means', 'dispersio

In [7]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['sampleID', 'sample_type_coarse', 'sample_type', 'processing_date',
       'dissociation_minutes', 'parity', 'brca_status', 'condition',
       'tissue_condition', 'reported_ethnicity', 'BMI', 'prob_spikein',
       'prob_spikein_dblt', 'pred_spikein', 'n_genes', 'percent_mito',
       'n_counts', 'level0_global', 'level1_global', 'level0', 'level1',
       'level2', 'scrublet_score', 'scrublet_cluster_score',
       'tissue_ontology_term_id', 'assay_ontology_term_id',
       'disease_ontology_term_id', 'cell_type_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id',
       'development_stage_ontology_term_id', 'sex_ontology_term_id',
       'donor_id', 'donor_age', 'risk_status', 'is_primary_data',
       'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease',
       'sex', 'tissue', 'self_reported_ethnicity', 'development_stage',
       'observation_joinid'],
      dtype='object')

In [9]:
# cell counts by cell type
print(adata_raw.obs["level2"].value_counts()) 

level2
LHS1                94935
FB1                 80168
FB2                 76725
LASP1               73109
VEV                 55113
LASP2               54842
LASP3               50491
BMYO1               30074
VEA                 29436
VEAT                28785
PV1                 27410
PV3                 22317
Doublet             20774
PV4                 20158
LHS2                18115
PV2                 14828
VEC                 14024
LHS3                11373
FB3                 11334
CD8_Trm              9426
LE1                  8407
FB4                  8241
LASP4                7976
stripped_nuclei      6899
PV5                  5534
LE2                  5349
CD8_Tc1              2984
Macro                2492
CD4_naive            1914
CD4_Th               1802
B_mem_switched       1619
BMYO2                1471
CD8_Tem              1075
B_naive               866
DDC1                  705
Plasma_cell           623
B_mem_unswitched      476
NKT                   282
DDC2 

In [10]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000186092', 'ENSG00000239945',
       'ENSG00000241860', 'ENSG00000241599', 'ENSG00000286448',
       'ENSG00000236601', 'ENSG00000235146', 'ENSG00000229905',
       'ENSG00000237491',
       ...
       'ENSG00000276345', 'ENSG00000277856', 'ENSG00000275063',
       'ENSG00000275869', 'ENSG00000273554', 'ENSG00000277836',
       'ENSG00000278633', 'ENSG00000276017', 'ENSG00000278817',
       'ENSG00000277196'],
      dtype='object', length=33604)
Index(['feature_types', 'genome', 'n_cells', 'highly_variable', 'means',
       'dispersions', 'dispersions_norm', 'gene_name', 'feature_is_filtered',
       'feature_name', 'feature_reference', 'feature_biotype',
       'feature_length', 'feature_type'],
      dtype='object')


In [12]:
# gene counts by feature type
print(adata_raw.var["feature_type"].value_counts())

feature_type
protein_coding                        18788
lncRNA                                14399
IG_V_gene                               127
TR_V_gene                                98
IG_V_pseudogene                          58
transcribed_unprocessed_pseudogene       27
TR_V_pseudogene                          19
transcribed_unitary_pseudogene           19
artifact                                 14
IG_C_gene                                14
TR_J_gene                                13
IG_C_pseudogene                           9
IG_J_gene                                 8
TR_C_gene                                 6
processed_pseudogene                      2
transcribed_processed_pseudogene          2
TEC                                       1
Name: count, dtype: int64


In [13]:
# defining cluster header
cluster_header = "level2" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [14]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,level2,count
0,LHS1,94935
1,FB1,80168
2,FB2,76725
3,LASP1,73109
4,VEV,55113
5,LASP2,54842
6,LASP3,50491
7,BMYO1,30074
8,VEA,29436
9,VEAT,28785


In [15]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells

unknown_col = "level2"          # column name in adata_raw.obs that contains the unknown labels
unknown_values = ["stripped_nuclei", "Doublet"]     # values in the unknown_col that indicate unknown cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unknown_col].isin(unknown_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_breast_ds_filter.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
              level2  count
0               LHS1  94935
1                FB1  80168
2                FB2  76725
3              LASP1  73109
4                VEV  55113
5              LASP2  54842
6              LASP3  50491
7              BMYO1  30074
8                VEA  29436
9               VEAT  28785
10               PV1  27410
11               PV3  22317
12               PV4  20158
13              LHS2  18115
14               PV2  14828
15               VEC  14024
16              LHS3  11373
17               FB3  11334
18           CD8_Trm   9426
19               LE1   8407
20               FB4   8241
21             LASP4   7976
22               PV5   5534
23               LE2   5349
24           CD8_Tc1   2984
25             Macro   2492
26         CD4_naive   1914
27            CD4_Th   1802
28    B_mem_switched   1619
29             BMYO2   1471
30           CD8_Tem   1075
31           B_naive    866
32   

## Heart (Downsample/Filter)

* **Original Dataset Size:** 704.2k cells

* **Number of Cell Types:** 70

* Filtering out `cell_state` where values are `unclassified`.

In [18]:
# load the dataset
file = "data_heart.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 704296 × 31832 backed at '../beckersv_data/data_heart.h5ad'
    obs: 'sangerID', 'donor_type', 'region', 'age', 'facility', 'flushed', 'cell_state', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'disease_ontology_term_id', 'is_primary_data', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'tissue_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'cell_state_colors', 'citation', 'default_embedding', 'donor_id_colors', 'donor_type_colors', 'facility_colors', 'flushed_colors', 'is_pre_an

In [19]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['sangerID', 'donor_type', 'region', 'age', 'facility', 'flushed',
       'cell_state', 'n_genes', 'n_genes_by_counts', 'total_counts',
       'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo',
       'pct_counts_ribo', 'scrublet_score', 'assay_ontology_term_id',
       'cell_type_ontology_term_id', 'development_stage_ontology_term_id',
       'donor_id', 'disease_ontology_term_id', 'is_primary_data',
       'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id',
       'suspension_type', 'tissue_ontology_term_id', 'tissue_type',
       'cell_type', 'assay', 'disease', 'sex', 'tissue',
       'self_reported_ethnicity', 'development_stage', 'observation_joinid'],
      dtype='object')

In [22]:
# cell counts by cell type
print(adata_raw.obs["cell_state"].value_counts()) 

cell_state
vCM1           97830
PC1_vent       55053
FB1            38709
vCM2           34692
FB2            32227
               ...  
gdT               21
NC3_glial         20
NC6_schwann       18
NC4_glial         13
CD4+T_Th17         7
Name: count, Length: 76, dtype: int64


In [23]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241599',
       'ENSG00000236601', 'ENSG00000284733', 'ENSG00000235146',
       'ENSG00000284662',
       ...
       'ENSG00000277196', 'ENSG00000277630', 'ENSG00000278384',
       'ENSG00000278633', 'ENSG00000276345', 'ENSG00000277856',
       'ENSG00000275063', 'ENSG00000271254', 'ENSG00000277475',
       'ENSG00000268674'],
      dtype='object', length=31832)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [26]:
# gene counts by feature type
print(adata_raw.var["feature_type"].value_counts())

feature_type
protein_coding                        19195
lncRNA                                11924
IG_V_pseudogene                         187
IG_V_gene                               146
TR_V_gene                               106
TR_J_gene                                79
IG_D_gene                                37
TR_V_pseudogene                          33
transcribed_unprocessed_pseudogene       29
IG_J_gene                                18
transcribed_unitary_pseudogene           17
artifact                                 17
IG_C_gene                                14
IG_C_pseudogene                           9
TR_C_gene                                 6
TR_J_pseudogene                           4
TR_D_gene                                 4
IG_J_pseudogene                           3
transcribed_processed_pseudogene          2
processed_pseudogene                      1
unprocessed_pseudogene                    1
Name: count, dtype: int64


In [27]:
# defining cluster header
cluster_header = "cell_state" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [28]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,cell_state,count
0,vCM1,97830
1,PC1_vent,55053
2,FB1,38709
3,vCM2,34692
4,FB2,32227
...,...,...
71,gdT,21
72,NC3_glial,20
73,NC6_schwann,18
74,NC4_glial,13


In [29]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells

unknown_col = "cell_state"          # column name in adata_raw.obs that contains the unknown labels
unknown_values = ["unclassified"]     # values in the unknown_col that indicate unknown cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unknown_col].isin(unknown_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_heart_ds_filter.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
     cell_state  count
0          vCM1  97830
1      PC1_vent  55053
2           FB1  38709
3          vCM2  34692
4           FB2  32227
..          ...    ...
70          gdT     21
71    NC3_glial     20
72  NC6_schwann     18
73    NC4_glial     13
74   CD4+T_Th17      7

[75 rows x 2 columns]
Ideal threshold found!
Ideal Threshold : 21118, New Total : 500518, Ideal Total : 500000.
New Counts:
0     21118
1     21118
2     21118
3     21118
4     21118
      ...  
70       21
71       20
72       18
73       13
74        7
Name: count, Length: 75, dtype: int64
Downsampling to maximum 21118 cells per cluster...
Returning indices of downsampled cells...


## Kidney (Filter)

* **Original Dataset Size:** 304.6k cells

* **Number of Cell Types:** 75

* Filtering out `disease` where values are `acute kidney injury` and `chronic kidney disease`.

In [31]:
# load the dataset
file = "data_kidney.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 304652 × 33140 backed at '../beckersv_data/data_kidney.h5ad'
    obs: 'nCount_RNA', 'nFeature_RNA', 'library', 'percent.er', 'percent.mt', 'degen.score', 'aEpi.score', 'aStr.score', 'cyc.score', 'matrisome.score', 'collagen.score', 'glycoprotein.score', 'proteoglycan.score', 'S.Score', 'G2M.Score', 'experiment', 'specimen', 'condition.long', 'condition.l1', 'condition.l2', 'donor_id', 'region.l1', 'region.l2', 'percent.cortex', 'percent.medulla', 'sample_tissue_type', 'id', 'pagoda_k100_infomap_coembed', 'subclass.full', 'subclass.l3', 'subclass.l2', 'subclass.l1', 'state.l2', 'state', 'class', 'structure', 'disease_ontology_term_id', 'sex_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'eGFR', 'BMI', 'diabetes_history', 'hypertension', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'dise

In [32]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['nCount_RNA', 'nFeature_RNA', 'library', 'percent.er', 'percent.mt',
       'degen.score', 'aEpi.score', 'aStr.score', 'cyc.score',
       'matrisome.score', 'collagen.score', 'glycoprotein.score',
       'proteoglycan.score', 'S.Score', 'G2M.Score', 'experiment', 'specimen',
       'condition.long', 'condition.l1', 'condition.l2', 'donor_id',
       'region.l1', 'region.l2', 'percent.cortex', 'percent.medulla',
       'sample_tissue_type', 'id', 'pagoda_k100_infomap_coembed',
       'subclass.full', 'subclass.l3', 'subclass.l2', 'subclass.l1',
       'state.l2', 'state', 'class', 'structure', 'disease_ontology_term_id',
       'sex_ontology_term_id', 'development_stage_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id', 'eGFR', 'BMI',
       'diabetes_history', 'hypertension', 'tissue_ontology_term_id',
       'assay_ontology_term_id', 'cell_type_ontology_term_id',
       'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type',
       'assay', 'diseas

In [33]:
# cell counts by cell type
print(adata_raw.obs["subclass.full"].value_counts()) 

subclass.full
Adaptive / Maladaptive / Repairing Proximal Tubule Epithelial Cell    26027
Degenerative Proximal Tubule Epithelial Cell                          25023
Cortical Thick Ascending Limb Cell                                    23767
Proximal Tubule Epithelial Cell Segment 1 / Segment 2                 17623
Adaptive / Maladaptive / Repairing Thick Ascending Limb Cell          14924
                                                                      ...  
Schwann Cell / Neural                                                    77
Plasmacytoid Dendritic Cell                                              75
Cycling Distal Convoluted Tubule Cell                                    75
Cycling Connecting Tubule Cell                                           68
Cycling Myofibroblast                                                    66
Name: count, Length: 75, dtype: int64


In [34]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000121410', 'ENSG00000268895', 'ENSG00000148584',
       'ENSG00000175899', 'ENSG00000245105', 'ENSG00000166535',
       'ENSG00000256661', 'ENSG00000184389', 'ENSG00000128274',
       'ENSG00000118017',
       ...
       'ENSG00000207468', 'ENSG00000201821', 'ENSG00000199477',
       'ENSG00000222489', 'ENSG00000242737', 'ENSG00000279622',
       'ENSG00000261499', 'ENSG00000279570', 'ENSG00000249263',
       'ENSG00000253460'],
      dtype='object', length=33140)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [35]:
# gene counts by feature type
print(adata_raw.var["feature_type"].value_counts())

feature_type
protein_coding                        18722
lncRNA                                11509
processed_pseudogene                   1084
transcribed_unprocessed_pseudogene      532
TEC                                     430
transcribed_processed_pseudogene        301
misc_RNA                                119
transcribed_unitary_pseudogene           90
IG_V_gene                                81
unprocessed_pseudogene                   65
TR_V_gene                                55
snRNA                                    33
Mt_tRNA                                  21
snoRNA                                   20
artifact                                 14
IG_C_gene                                13
IG_V_pseudogene                          12
TR_V_pseudogene                           7
TR_C_gene                                 6
scaRNA                                    5
miRNA                                     5
IG_C_pseudogene                           4
unitary_pseudogene 

In [36]:
# defining cluster header
cluster_header = "subclass.full" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [37]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,subclass.full,count
0,Adaptive / Maladaptive / Repairing Proximal Tu...,26027
1,Degenerative Proximal Tubule Epithelial Cell,25023
2,Cortical Thick Ascending Limb Cell,23767
3,Proximal Tubule Epithelial Cell Segment 1 / Se...,17623
4,Adaptive / Maladaptive / Repairing Thick Ascen...,14924
...,...,...
70,Schwann Cell / Neural,77
71,Plasmacytoid Dendritic Cell,75
72,Cycling Distal Convoluted Tubule Cell,75
73,Cycling Connecting Tubule Cell,68


In [38]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells

unhealthy_col = "disease"      # column name in adata_raw.obs that contains the unhealthy labels
unhealthy_values = ["acute kidney injury", "chronic kidney disease"] # values in the unhealthy_col that indicate unhealthy cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unhealthy_col].isin(unhealthy_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = output_folder + "data_kidney_ds_filter.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
                                        subclass.full  count
0   Proximal Tubule Epithelial Cell Segment 1 / Se...   8551
1        Degenerative Proximal Tubule Epithelial Cell   7586
2                  Cortical Thick Ascending Limb Cell   7354
3   Adaptive / Maladaptive / Repairing Proximal Tu...   5872
4                                Medullary Fibroblast   4589
..                                                ...    ...
70                        Plasmacytoid Dendritic Cell     31
71                              Cycling Myofibroblast     18
72  Cycling Natural Killer Cell / Natural Killer T...     16
73                     Cycling Connecting Tubule Cell     12
74              Cycling Distal Convoluted Tubule Cell      6

[75 rows x 2 columns]
Downsampling to maximum 500000 cells per cluster...
Returning indices of downsampled cells...


## Lung (Downsample)

* **Original Dataset Size:** 584.9k cells

* **Number of Cell Types:** 61

In [96]:
# load the dataset
file = "data_lung.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 584944 × 27402 backed at '../beckersv_data/data_lung.h5ad'
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_

In [97]:
adata_raw.X

CSRDataset: backend hdf5, shape (584944, 27402), data_dtype float32

In [43]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['suspension_type', 'donor_id', 'is_primary_data',
       'assay_ontology_term_id', 'cell_type_ontology_term_id',
       'development_stage_ontology_term_id', 'disease_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id',
       'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range',
       'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling',
       'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3',
       'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset',
       'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3',
       'entropy_original_ann_level_2_clean_leiden_3',
       'entropy_original_ann_level_3_clean_leiden_3',
       'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1',
       'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts',
       'lung_condition', 'mixed_ancestry', 'n_genes_detected',
       'original_ann_highest_res', 'original_

In [44]:
# cell counts by cell type
print(adata_raw.obs["ann_finest_level"].value_counts()) 

ann_finest_level
Alveolar macrophages          68487
AT2                           61429
Suprabasal                    41158
Basal resting                 38955
Goblet (nasal)                35833
                              ...  
Mesothelium                     230
Tuft                            165
Neuroendocrine                  159
Hematopoietic stem cells         60
Lymphatic EC proliferating       28
Name: count, Length: 61, dtype: int64


In [45]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167',
       ...
       'ENSG00000283052', 'ENSG00000283063', 'ENSG00000283064',
       'ENSG00000283071', 'ENSG00000283075', 'ENSG00000283078',
       'ENSG00000283103', 'ENSG00000283117', 'ENSG00000283118',
       'ENSG00000283125'],
      dtype='object', length=27402)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [47]:
# gene counts by feature type
print(adata_raw.var["feature_type"].value_counts())

feature_type
protein_coding                        18178
lncRNA                                 8744
IG_V_gene                               120
TR_V_gene                               100
transcribed_unprocessed_pseudogene       91
transcribed_unitary_pseudogene           44
transcribed_processed_pseudogene         25
TEC                                      14
IG_C_gene                                13
artifact                                 13
TR_J_gene                                12
IG_V_pseudogene                          11
TR_V_pseudogene                           9
processed_pseudogene                      7
IG_C_pseudogene                           7
TR_C_gene                                 6
IG_J_gene                                 6
translated_processed_pseudogene           1
unprocessed_pseudogene                    1
Name: count, dtype: int64


In [48]:
# defining cluster header
cluster_header = "ann_finest_level" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [49]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,ann_finest_level,count
0,Alveolar macrophages,68487
1,AT2,61429
2,Suprabasal,41158
3,Basal resting,38955
4,Goblet (nasal),35833
...,...,...
56,Mesothelium,230
57,Tuft,165
58,Neuroendocrine,159
59,Hematopoietic stem cells,60


In [50]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if not applicable, just downsample by cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, output_folder, "data_lung_ds.h5ad", return_index=False, write_to_file=True)

Finding threshold for downsampling to 500000 cells...
Original Counts:
              ann_finest_level  count
0         Alveolar macrophages  68487
1                          AT2  61429
2                   Suprabasal  41158
3                Basal resting  38955
4               Goblet (nasal)  35833
..                         ...    ...
56                 Mesothelium    230
57                        Tuft    165
58              Neuroendocrine    159
59    Hematopoietic stem cells     60
60  Lymphatic EC proliferating     28

[61 rows x 2 columns]
Ideal threshold found!
Ideal Threshold : 31248, New Total : 491345, Ideal Total : 500000.
New Counts:
0     31248
1     31248
2     31248
3     31248
4     31248
      ...  
56      230
57      165
58      159
59       60
60       28
Name: count, Length: 61, dtype: int64
Downsampling to maximum 31248 cells per cluster...
Writing downsampled anndata object to ../beckersv_data/data_lung_ds.h5ad...
